# Mesh convergence -- $N_x=1$ (X-point at $(1.45,-1.30)$), five meshes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BecerraMiguel/SemiFree-Solver/blob/main/notebooks/03_mesh_convergence_Nx1.ipynb)

The same refinement study as in the previous notebook, but with a single X-point at
$(R_X,Z_X)=(1.45, -1.30)$ m, solved as an equality-constrained least-squares problem (null-space method:
$B_R=B_Z=0$ and $\psi=\psi_{ref}$ at the X-point).

**Inputs.** For every mesh, `cases/DIII-D_<mesh>/` contains the boundary deformed to create the X-point
(`Dshape_xpoint.txt`, produced with the $C^1$ boundary transformation in `src/semifree_boundary/xpoint_transform_c1.py`)
and the current density consistent with that boundary (`Jt_xpoint.txt`, produced with the polygonal-boundary
fixed-boundary solver `src/grad_shafranov/main_poly.cpp`). The semi-free solver is run on them with
boundary-transformation method 0 (the boundary is not transformed again).

**Analysis:** (i) $\psi$, $B_R$ and $B_Z$ evaluated exactly at the X-point; (ii) convergence of $\psi$ in three
regions; (iii) convergence of the coil currents; (iv) run times. As in the $N_x=0$ notebook, the dense
$B_R$/$B_Z$ sweeps are not computed.

**Estimated time (Colab, 2 cores):** ~1.5 min build + 1.1, 1.7, 4.6, 12.8 and 40.6 min per mesh =
**~1 hour**. Quick mode: ~10 minutes.


**DIII-D case** (same for every mesh): $R_0=1.67$ m, $a=0.67$ m, $\kappa=1.77$, $\delta=0.30$,
$I_p=1.5$ MA, $P_{axis}=50$ kPa, $B_{axis}=2.0$ T, $\Psi_b=0$, with $N_c=18$ PF coils.
Computational domain: $R\in[0.15, 3.0]$ m, $Z\in[-1.75, 1.75]$ m.

The inputs ($J_\phi$, boundary, coil positions) are shipped with the repository in `cases/DIII-D_<mesh>/`, and the
solver configuration in `configs/DIII-D_<mesh>.json`. This notebook clones the repository, builds the solver and
runs everything from there: nothing has to be uploaded.


**Options (first code cell):**
- `USE_GOOGLE_DRIVE = True` keeps the results in your Drive: if Colab disconnects, re-running the notebook reuses
the runs that already finished, and the notebooks share results with each other.
- `QUICK_MODE = True` uses only the 82x142, 100x172 and 151x261 meshes.

The values labelled *reference* are those obtained in earlier runs of the same study on Colab (2 cores). Timings
depend on the hardware; $\psi$ and the coil currents should agree with the reference up to rounding.



In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/BecerraMiguel/SemiFree-Solver.git'
REPO_REF = None        # branch or tag to clone (None = default branch)
REPO_DIR = os.environ.get('REPRODUCE_REPO_DIR', '/content/SemiFree-Solver')
if not os.path.isdir(REPO_DIR):
    cmd = ['git', 'clone', '--depth', '1'] + (['--branch', REPO_REF] if REPO_REF else [])
    subprocess.run(cmd + [REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f'{REPO_DIR}/notebooks')
import reproduce_common as rc
import numpy as np
import matplotlib.pyplot as plt

USE_GOOGLE_DRIVE = False   # True: keep results in Google Drive (survive disconnections, shared between notebooks)
QUICK_MODE = False       # True: only the three coarsest meshes (fast test; the order fits then use 2 points)
WORK = rc.default_work_dir(USE_GOOGLE_DRIVE)
os.makedirs(f'{WORK}/figures', exist_ok=True)
TAGS = rc.QUICK_TAGS if QUICK_MODE else rc.MESH_TAGS
print('Meshes:', TAGS)
rc.environment_report()
print('Working directory:', WORK)

## 1. Build

In [ ]:
BIN = rc.build_semifree(REPO_DIR, WORK, skip_bfield=True)
rc.mesh_table(REPO_DIR, TAGS)
print('X-point (R, Z) =', rc.XPOINT_RZ)

## 2. Run the solver on each mesh ($N_x=1$)

In [ ]:
for tag in TAGS:
    rc.run_semifree(BIN, REPO_DIR, WORK, 'Nx1', tag)

## 3. Load results and run times

In [ ]:
RUNS = rc.load_runs(REPO_DIR, WORK, 'Nx1', TAGS)
rc.print_timing_table(RUNS, TAGS, 'Nx1')

## 4. Conditions at the X-point
$\psi$, $B_R$ and $B_Z$ evaluated exactly at $(1.45,-1.30)$: $\psi$ and $B_R$ are essentially zero on every mesh and $B_Z$ decreases under refinement.

In [ ]:
rc.xpoint_table(RUNS, TAGS)

## 5. Solution on the 151x261 mesh
Poloidal flux over the whole domain (the cyan curve is the $\psi=0$ isocontour, which passes through the X-point), a zoom at the X-point, and $B_R$, $B_Z$ along $Z$ at $R=1.45$ m (from the derivative of $\psi$).

In [ ]:
if '151x261' in RUNS:
    rc.plot_xpoint_flux(RUNS['151x261'], fname=f'{WORK}/figures/nx1_flux_151x261.png')
    plt.show()
    rc.plot_xpoint_zoom(RUNS['151x261'], fname=f'{WORK}/figures/nx1_zoom_151x261.png')
    plt.show()
    rc.plot_xpoint_field(RUNS['151x261'], fname=f'{WORK}/figures/nx1_field_151x261.png')
    plt.show()
else:
    print('The 151x261 mesh is not available.')

## 6. Convergence of $\psi$

In [ ]:
PSI = rc.psi_convergence(REPO_DIR, RUNS, TAGS)
rc.print_psi_table(PSI, 'Nx1')
rc.plot_psi_convergence(PSI, 'Nx=1', fname=f'{WORK}/figures/nx1_psi_convergence.png')
plt.show()

## 7. Convergence of the coil currents
With the X-point the convergence is noisier than without it.

In [ ]:
COIL = rc.coil_convergence(REPO_DIR, RUNS, TAGS)
rc.print_coil_table(COIL, 'Nx1')

## 8. Run times
The $N_x=0$ curve is included if notebook 02 was run on the same working directory (for example through Google Drive).

In [ ]:
RUNS0 = rc.load_runs(REPO_DIR, WORK, 'Nx0', TAGS)
series = {'Nx=1 (this run)': {t: r['timing']['psi_min'] for t, r in RUNS.items()},
          'Nx=1 (reference, Colab)': rc.REFERENCE['psi_minutes']['Nx1'],
          'Nx=0 (reference, Colab)': rc.REFERENCE['psi_minutes']['Nx0']}
if RUNS0:
    series['Nx=0 (this run)'] = {t: r['timing']['psi_min'] for t, r in RUNS0.items()}
rc.plot_timing(REPO_DIR, series, fname=f'{WORK}/figures/nx1_timing.png')
plt.show()

## 9. Results bundle (optional)

In [ ]:
import glob
zp = rc.zip_results(WORK, ['Nx1'], f'{WORK}/results_mesh_convergence_Nx1.zip',
                    extra_files=sorted(glob.glob(f'{WORK}/figures/nx1_*.png')))
rc.offer_download(zp)